# Project 09 -- CVA and Counterparty Credit Risk

**Credit Valuation Adjustment** (CVA) is the market price of counterparty credit risk.  This notebook walks through the full CVA pipeline:

1. Simulate interest rate paths (Vasicek model)
2. Mark-to-market a vanilla interest rate swap
3. Compute exposure profiles (EE, PFE)
4. Bootstrap default probabilities and compute CVA
5. Netting analysis: gross vs netted CVA
6. Wrong-way risk sensitivity

**Reference:** Gregory (2020), *The xVA Challenge*, 4th ed., Wiley.

In [ ]:
import sys
from pathlib import Path

# Add project src to path
_PROJECT_SRC = str(Path.cwd().parent / "src")
if _PROJECT_SRC not in sys.path:
    sys.path.insert(0, _PROJECT_SRC)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

# Project modules
from instruments import InterestRateSwap, simulate_rate_paths
from credit import (
    hazard_rate_from_cds, survival_probability, default_probability,
    bootstrap_hazard_rates, marginal_default_prob,
)
from exposure import compute_exposure_profiles, apply_netting, apply_collateral
from cva import compute_cva, compute_dva, compute_bilateral_cva, cva_by_netting_set
from model import CVAModel
from diagnostics import (
    plot_exposure_profiles, plot_cva_waterfall,
    plot_netting_benefit, plot_wrong_way_risk,
)

# Load config
with open(Path.cwd().parent / "configs" / "default.yaml") as f:
    config = yaml.safe_load(f)

print("Config loaded successfully.")
print(f"Simulation: {config['simulation']['n_paths']} paths, "
      f"{config['simulation']['n_steps']} steps over {config['swap']['tenor']} years")

## 1. Simulate Vasicek Rate Paths

The **Vasicek model** for the short rate:

$$dr_t = \kappa(\theta - r_t)\,dt + \sigma\,dW_t$$

Key properties:
- **Mean-reverting** to $\theta$ at speed $\kappa$
- **Gaussian** process (rates can go negative, acceptable for this analysis)
- Analytical tractability for bond pricing

We simulate using Euler-Maruyama discretization and plot a fan chart showing the distribution of rate paths over time.

In [ ]:
ir = config["interest_rate"]
sw = config["swap"]
sim = config["simulation"]
seed = config["random_seed"]

# Simulate Vasicek paths
rate_paths, times = simulate_rate_paths(
    r0=ir["r0"], kappa=ir["kappa"], theta=ir["theta"], sigma=ir["sigma"],
    T=sw["tenor"], n_steps=sim["n_steps"], n_paths=sim["n_paths"], seed=seed,
)

print(f"Rate paths shape: {rate_paths.shape}")
print(f"Time grid: {times[0]:.1f} to {times[-1]:.1f} years, {len(times)} points")
print(f"Initial rate: {rate_paths[0, 0]:.4f}")
print(f"Final rate mean: {np.mean(rate_paths[:, -1]):.4f} (theta = {ir['theta']})")

# Fan chart
fig, ax = plt.subplots(figsize=(10, 6))
percentiles = [1, 5, 25, 50, 75, 95, 99]
colors_fan = plt.cm.Blues(np.linspace(0.2, 0.8, len(percentiles) // 2))

for i in range(len(percentiles) // 2):
    lo = np.percentile(rate_paths, percentiles[i], axis=0)
    hi = np.percentile(rate_paths, percentiles[-(i + 1)], axis=0)
    ax.fill_between(times, lo, hi, alpha=0.3, color=colors_fan[i],
                    label=f"{percentiles[i]}%-{percentiles[-(i+1)]}%")

median = np.percentile(rate_paths, 50, axis=0)
ax.plot(times, median, color="darkblue", linewidth=2, label="Median")
ax.axhline(ir["theta"], color="red", linestyle="--", linewidth=1, alpha=0.7, label=f"theta = {ir['theta']}")

ax.set_xlabel("Time (years)")
ax.set_ylabel("Short Rate")
ax.set_title("Vasicek Short Rate Paths -- Fan Chart")
ax.legend(loc="upper right")
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## 2. Mark-to-Market Swap Values

We price a **receiver interest rate swap** (receive fixed, pay floating) at each simulation time step.  The mark-to-market value:

$$V(t) = N \cdot (K - r(t)) \cdot A(t)$$

where $N$ is the notional, $K$ is the fixed rate, $r(t)$ is the floating rate at time $t$, and $A(t)$ is the remaining annuity factor.

The swap value is:
- **Positive** when fixed rate > floating rate (counterparty owes us)
- **Negative** when fixed rate < floating rate (we owe counterparty)

In [ ]:
# Create swap and compute MTM values
swap = InterestRateSwap(
    notional=sw["notional"], fixed_rate=sw["fixed_rate"],
    tenor=sw["tenor"], payment_freq=sw["payment_freq"], seed=seed,
)
mtm_values = swap.simulate_values(rate_paths, times)

print(f"MTM values shape: {mtm_values.shape}")
print(f"MTM at t=0: mean={np.mean(mtm_values[:, 0]):,.0f}")
print(f"MTM at t=2.5: mean={np.mean(mtm_values[:, 10]):,.0f}, "
      f"std={np.std(mtm_values[:, 10]):,.0f}")

# Distribution of MTM at selected time points
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
time_indices = [4, 10, 16]  # ~1y, ~2.5y, ~4y

for ax, idx in zip(axes, time_indices):
    t = times[idx]
    vals = mtm_values[:, idx]
    ax.hist(vals, bins=50, color="steelblue", alpha=0.7, edgecolor="white")
    ax.axvline(0, color="red", linestyle="--", linewidth=1, alpha=0.7)
    ax.axvline(np.mean(vals), color="darkblue", linewidth=2, label=f"Mean: {np.mean(vals):,.0f}")
    ax.set_xlabel("MTM Value")
    ax.set_ylabel("Count")
    ax.set_title(f"Swap MTM Distribution at t = {t:.1f}y")
    ax.legend()
    ax.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

## 3. Exposure Profiles (EE, PFE)

Key exposure metrics for counterparty credit risk:

- **Expected Exposure (EE):** $\text{EE}(t) = \mathbb{E}[\max(V(t), 0)]$ -- the average positive exposure across scenarios.
- **Potential Future Exposure (PFE):** $\text{PFE}_\alpha(t)$ -- the $\alpha$-quantile of the positive exposure distribution.  PFE at 97.5% is a common regulatory measure.
- **Expected Positive Exposure (EPE):** time-weighted average of EE, used in regulatory capital calculations.
- **Expected Negative Exposure (ENE):** $\text{ENE}(t) = \mathbb{E}[\min(V(t), 0)]$ -- relevant for DVA.

The exposure profile of an interest rate swap typically:
- Starts at zero (at-the-money at inception)
- Rises as rate uncertainty grows (diffusion effect)
- Falls as time-to-maturity decreases (amortisation effect)

This gives the characteristic "hump-shaped" profile.

In [ ]:
# Compute exposure profiles
profiles = compute_exposure_profiles(mtm_values, times)

print(f"EPE (time-weighted avg EE): {profiles['epe']:,.0f}")
print(f"Peak EE: {np.max(profiles['ee']):,.0f} at t = {times[np.argmax(profiles['ee'])]:.2f}y")
print(f"Peak PFE(97.5%): {np.max(profiles['pfe_975']):,.0f} at t = {times[np.argmax(profiles['pfe_975'])]:.2f}y")

# Plot exposure profiles
fig, ax = plot_exposure_profiles(profiles, times)
plt.show()

# Summary table
summary_df = pd.DataFrame({
    "Time": times,
    "EE": profiles["ee"],
    "PFE (97.5%)": profiles["pfe_975"],
    "PFE (99%)": profiles["pfe_99"],
    "ENE": profiles["ene"],
})
print("\nExposure Summary (selected times):")
display(summary_df.iloc[::4].round(0).reset_index(drop=True))

## 4. Default Probabilities and CVA Computation

### Hazard Rate Bootstrapping

From CDS spreads, we extract the implied hazard rate:

$$\lambda = \frac{s}{1 - R}$$

where $s$ is the CDS spread and $R$ is the recovery rate.

### CVA Formula

Unilateral CVA is the expected loss due to counterparty default:

$$\text{CVA} = (1 - R) \sum_{i=1}^{N} \text{EE}(t_i) \cdot [\text{PD}(t_i) - \text{PD}(t_{i-1})] \cdot D(t_i)$$

**Bilateral CVA** accounts for both counterparty and own default:
$$\text{BCVA} = \text{CVA} - \text{DVA}$$

In [ ]:
cpty = config["counterparty"]
own = config["own_credit"]

# Hazard rates
cpty_hazard = hazard_rate_from_cds(cpty["cds_spread"], cpty["recovery"])
own_hazard = hazard_rate_from_cds(own["cds_spread"], own["recovery"])
print(f"Counterparty hazard rate: {cpty_hazard:.4f} (from {cpty['cds_spread']*10000:.0f} bps CDS)")
print(f"Own hazard rate: {own_hazard:.4f} (from {own['cds_spread']*10000:.0f} bps CDS)")

# Survival and default probabilities over time
t_grid = np.linspace(0, 5, 100)
surv_cpty = [survival_probability(cpty_hazard, t) for t in t_grid]
pd_cpty = [default_probability(cpty_hazard, t) for t in t_grid]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(t_grid, surv_cpty, color="steelblue", linewidth=2)
ax1.set_xlabel("Time (years)")
ax1.set_ylabel("Q(t)")
ax1.set_title("Counterparty Survival Probability")
ax1.grid(True, alpha=0.3)

ax2.plot(t_grid, pd_cpty, color="firebrick", linewidth=2)
ax2.set_xlabel("Time (years)")
ax2.set_ylabel("PD(t)")
ax2.set_title("Counterparty Cumulative Default Probability")
ax2.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

# Bootstrap from term structure
cds_term_structure = {1.0: 0.008, 2.0: 0.009, 3.0: 0.010, 5.0: 0.012}
bootstrapped = bootstrap_hazard_rates(cds_term_structure, cpty["recovery"])
print("\nBootstrapped hazard rates from CDS term structure:")
for tenor, hr in bootstrapped.items():
    print(f"  {tenor:.0f}Y: lambda = {hr:.6f}")

# Compute bilateral CVA
result = compute_bilateral_cva(
    ee=profiles["ee"], ene=profiles["ene"], times=times,
    cpty_hazard=cpty_hazard, own_hazard=own_hazard, recovery=cpty["recovery"],
)
print(f"\n--- CVA Results ---")
print(f"CVA:  {result['cva']:>12,.2f}")
print(f"DVA:  {result['dva']:>12,.2f}")
print(f"BCVA: {result['bcva']:>12,.2f}")
print(f"CVA as % of notional: {result['cva'] / sw['notional'] * 100:.4f}%")

# Waterfall chart
fig, ax = plot_cva_waterfall(result)
plt.show()

## 5. Netting Analysis: Gross vs Net CVA

**Netting** is a legal agreement where, if a counterparty defaults, the net exposure across all trades in the netting set is used rather than the sum of individual positive exposures.

Without netting (gross): $\text{Exposure} = \sum_i \max(V_i, 0)$

With netting: $\text{Exposure} = \max\left(\sum_i V_i, 0\right)$

Netting benefit arises because negative-value trades offset positive-value trades.  We create 3 swaps with different fixed rates to illustrate diversification within a netting set.

In [ ]:
# Create netting set with 3 swaps of different fixed rates
netting_cfg = config["netting"]
n_trades = netting_cfg["n_trades"]
offsets = np.linspace(-0.005, 0.005, n_trades)

trades_mtm = []
trade_labels = []
for i, offset in enumerate(offsets):
    fixed = sw["fixed_rate"] + offset
    swap_i = InterestRateSwap(
        notional=sw["notional"], fixed_rate=fixed,
        tenor=sw["tenor"], payment_freq=sw["payment_freq"], seed=seed + i,
    )
    mtm_i = swap_i.simulate_values(rate_paths, times)
    trades_mtm.append(mtm_i)
    trade_labels.append(f"Swap {i+1} (K={fixed:.3f})")

# Netting analysis
netting_result = cva_by_netting_set(trades_mtm, times, cpty_hazard, cpty["recovery"])

print("=== Netting Analysis ===")
print(f"Gross CVA (no netting):   {netting_result['gross_cva']:>12,.2f}")
print(f"Net CVA (with netting):   {netting_result['net_cva']:>12,.2f}")
print(f"Netting benefit:          {netting_result['netting_benefit']:>12,.2f}")
print(f"Benefit percentage:       {netting_result['benefit_pct']:>11.1f}%")

# Use CVAModel for netting DataFrame
model = CVAModel(config)
model._rate_paths = rate_paths
model._times = times
model._mtm_values = mtm_values
model._profiles = profiles
netting_df = model.netting_analysis()
display(netting_df)

# Plot
fig, ax = plot_netting_benefit(netting_df)
plt.show()

# Show individual vs netted exposure profiles
fig, ax = plt.subplots(figsize=(10, 6))
for i, (mtm_i, label) in enumerate(zip(trades_mtm, trade_labels)):
    ee_i = np.mean(np.maximum(mtm_i, 0.0), axis=0)
    ax.plot(times, ee_i, linestyle="--", alpha=0.7, label=f"EE {label}")

netted = apply_netting(trades_mtm)
ee_net = np.mean(netted, axis=0)
ax.plot(times, ee_net, linewidth=2.5, color="darkblue", label="EE (Netted)")

ax.set_xlabel("Time (years)")
ax.set_ylabel("Expected Exposure")
ax.set_title("Individual vs Netted Exposure Profiles")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## 6. Wrong-Way Risk Sensitivity

**Wrong-way risk (WWR)** occurs when the exposure to a counterparty increases when the counterparty's credit quality deteriorates.  Formally, it is positive correlation between exposure and default probability.

We model WWR by adjusting the hazard rate based on the exposure level:

$$\lambda_{\text{adj}}(t) = \lambda \cdot \exp(\rho \cdot z(t))$$

where $\rho$ is the correlation parameter and $z(t)$ is the standardised exposure.

- $\rho > 0$: **Wrong-way risk** -- CVA increases
- $\rho = 0$: **Independent** (baseline)
- $\rho < 0$: **Right-way risk** -- CVA decreases

In [ ]:
# Wrong-way risk analysis
correlations = config["wrong_way_risk"]["correlations"]

# Use CVAModel to compute WWR CVA at each correlation level
model = CVAModel(config)
model._rate_paths = rate_paths
model._times = times
model._mtm_values = mtm_values
model._profiles = profiles

wwr_cvas = []
for rho in correlations:
    wwr_result = model.wrong_way_risk(rho)
    wwr_cvas.append(wwr_result["cva"])

# Display results
print("=== Wrong-Way Risk Sensitivity ===")
print(f"{'Correlation':>12} {'CVA':>12} {'Change vs rho=0':>16}")
print("-" * 44)
baseline_idx = correlations.index(0.0)
baseline_cva = wwr_cvas[baseline_idx]
for rho, cva_val in zip(correlations, wwr_cvas):
    change = (cva_val / baseline_cva - 1) * 100 if baseline_cva > 0 else 0
    print(f"{rho:>12.2f} {cva_val:>12,.2f} {change:>15.1f}%")

# Plot
fig, ax = plot_wrong_way_risk(correlations, wwr_cvas)
plt.show()

print("\nKey takeaway: Wrong-way risk (positive correlation) significantly")
print("increases CVA, while right-way risk (negative correlation) provides")
print("a natural hedge against counterparty default losses.")